# fit new model

colors trial type and their indices:


yellow = 1   ---> 0

orange = 2   ---> 1

red = 3      ---> 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
os.environ["OMP_NUM_THREADS"] = "1"
import seaborn as sns 
from scipy.io import loadmat
import ast
from scipy.io import loadmat, savemat
import warnings
warnings.filterwarnings("ignore")
import statsmodels.api as sm


In [ ]:
outputFolderName = r"\\155.100.91.44\d\Data\Nill\BART_param_recovery\new_modeling\param_recovery_1_data"
inputFolderName = r"\\155.100.91.44\d\Data\Nill\BART_param_recovery\old_modeling\param_recovery_3_simulated_fields"

if not os.path.exists(outputFolderName):
    os.makedirs(outputFolderName)

In [ ]:


matFiles = [f for f in os.listdir(inputFolderName) if f.endswith(".mat")]
nPatients = len(matFiles)

boundary_neg_count = 0

for pt in range(nPatients):
    fileName = matFiles[pt]

    ptID = os.path.splitext(fileName)[0]
    ptID = ptID.replace("_TDdataParamRecovery", "")

    print(f"\nprocessing pt {pt+1}/{nPatients}: {ptID}")

    matFile = os.path.join(inputFolderName, fileName)
    mat = loadmat(matFile, struct_as_record=False, squeeze_me=True)
    TDdataParamRecovery = mat["TDdataParamRecovery"]

    alphas = np.asarray(TDdataParamRecovery.a, dtype=float).ravel()
    nAlpha = len(alphas)

    # real fields for model fitting
    reward = np.asarray(TDdataParamRecovery.Reward, dtype=float).ravel()

    result_raw = np.asarray(TDdataParamRecovery.result, dtype=str)
    result_raw = np.array([str(x).strip().lower() for x in np.ravel(result_raw)])
    y = np.array([1 if x == "banked" else 0 for x in result_raw], dtype=float)

    trial_type = np.asarray(TDdataParamRecovery.trial_type, dtype=int).ravel()

    nTrials = int(TDdataParamRecovery.nTrials)

    reward = reward[:nTrials]
    y = y[:nTrials]
    trial_type = trial_type[:nTrials]

    # keep trial types 1,2,3,4
    valid_mask = np.isin(trial_type, [1, 2, 3, 4])

    reward_valid = reward[valid_mask]
    y_valid = y[valid_mask]
    trial_type_valid = trial_type[valid_mask]

    nTrials_valid = len(reward_valid)
    nTypes = 4

    if nTrials_valid == 0:
        print(f"{ptID}: no valid trials after filtering, skipping")
        continue

    fit_score = np.full((nAlpha, nAlpha), np.nan)
    expectedReward_1d_all = np.full((nAlpha, nAlpha, nTrials_valid), np.nan, dtype=float)
    RewardPE_1d_all = np.full((nAlpha, nAlpha, nTrials_valid), np.nan, dtype=float)

    for ap in range(nAlpha - 1, -1, -1):
        for an in range(nAlpha - 1, -1, -1):

            # one expected value trace per balloon color / trial type
            expectedReward = np.full((nTypes, nTrials_valid), np.nan, dtype=float)
            RewardPE = np.full((nTypes, nTrials_valid), np.nan, dtype=float)

            expectedReward[:, 0] = 0.0
            RewardPE[:, 0] = 0.0

            # trialwise values for the current trial's color only
            expectedReward_1d = np.full(nTrials_valid, np.nan, dtype=float)
            RewardPE_1d = np.full(nTrials_valid, np.nan, dtype=float)

            first_type = trial_type_valid[0] - 1
            expectedReward_1d[0] = expectedReward[first_type, 0]
            RewardPE_1d[0] = RewardPE[first_type, 0]

            for t in range(1, nTrials_valid):
                # carry forward all color values
                expectedReward[:, t] = expectedReward[:, t - 1]

                tt = trial_type_valid[t] - 1

                # PE for current trial's balloon color
                RewardPE[tt, t] = reward_valid[t] - expectedReward[tt, t - 1]

                # update only current color
                if RewardPE[tt, t] > 0:
                    expectedReward[tt, t] = expectedReward[tt, t - 1] + alphas[ap] * RewardPE[tt, t]
                elif RewardPE[tt, t] < 0:
                    expectedReward[tt, t] = expectedReward[tt, t - 1] + alphas[an] * RewardPE[tt, t]
                else:
                    expectedReward[tt, t] = expectedReward[tt, t - 1]

                expectedReward_1d[t] = expectedReward[tt, t]
                RewardPE_1d[t] = RewardPE[tt, t]

            expectedReward_1d_all[ap, an, :] = expectedReward_1d
            RewardPE_1d_all[ap, an, :] = RewardPE_1d

            mask_glm = np.isfinite(expectedReward_1d) & np.isfinite(y_valid)
            x_glm = expectedReward_1d[mask_glm]
            y_glm = y_valid[mask_glm]

            if len(y_glm) == 0 or len(np.unique(y_glm)) < 2:
                fit_score[ap, an] = np.nan
                continue

            X = sm.add_constant(x_glm, has_constant="add")

            try:
                model = sm.GLM(
                    y_glm,
                    X,
                    family=sm.families.Binomial(link=sm.families.links.Logit())
                )
                glm_result = model.fit()
                fit_score[ap, an] = glm_result.llf
            except Exception:
                fit_score[ap, an] = np.nan

    if np.all(np.isnan(fit_score)):
        print(f"{ptID}: all fit scores are NaN, skipping save")
        continue

    bestAlphaPosIdx, bestAlphaNegIdx = np.unravel_index(np.nanargmax(fit_score), fit_score.shape)

    bestAlphaPos = alphas[bestAlphaPosIdx]
    bestAlphaNeg = alphas[bestAlphaNegIdx]

    bestExpectedReward = expectedReward_1d_all[bestAlphaPosIdx, bestAlphaNegIdx, :]
    bestRewardPE = RewardPE_1d_all[bestAlphaPosIdx, bestAlphaNegIdx, :]

    print(f"{ptID}: bestAlphaPos = {bestAlphaPos}")
    print(f"{ptID}: bestAlphaNeg = {bestAlphaNeg}")


    if bestAlphaNegIdx == nAlpha - 1:
        print(f"{ptID}: WARNING - bestAlphaNeg is at the upper boundary ({bestAlphaNeg})")
        boundary_neg_count += 1

    td_dict = {}
    for key in TDdataParamRecovery._fieldnames:
        if key not in ["bestAlphaPos", "bestAlphaNeg", "bestExpectedReward", "bestRewardPE", "fit_score"]:
            td_dict[key] = getattr(TDdataParamRecovery, key)

    td_dict["bestAlphaPos"] = bestAlphaPos
    td_dict["bestAlphaNeg"] = bestAlphaNeg
    td_dict["bestExpectedReward"] = bestExpectedReward
    td_dict["bestRewardPE"] = bestRewardPE
    td_dict["fit_score"] = fit_score

    save_file_name = f"{ptID}_TDdataParamRecovery.mat"
    save_path = os.path.join(outputFolderName, save_file_name)

    savemat(save_path, {"TDdataParamRecovery": td_dict})



# debug

In [ ]:
fields = [f for f in dir(TDdataParamRecovery) if not f.startswith('_')]
print(fields)